In [1]:
import pandas as pd
import duckdb

# Transform and Load Data in pandas & duckdb (postgreSQL)

Note: if it were a data engineering oriented mission, I would orchestrate this with airflow and dbt. 
For simplicity the pipeline is here only in this notebook.

In [3]:
sales_df = pd.read_csv("data/sales_data_sample_utf.csv")

In [4]:
sales_df.head()

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


In [5]:
sales_df.columns

Index(['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER',
       'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID',
       'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE',
       'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE',
       'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME',
       'DEALSIZE'],
      dtype='str')

## Organising data according to a Star Schema

Currently, all informations are stored in one unique table. This is not efficient (even though we only have ~2K rows, this would become a problem at a larger scale). It is also bad practice to keep personal details in the same table as the sales data. 

Let us organise the data differently, as we would in a real-life setting, following good practice. For simplicity, we will use a [star schema](https://medium.com/@StefanoMeloccaro/data-warehouse-schemas-explained-star-snowflake-galaxy-d2c508d9ac5f) and differenciate fact and dimension tables.

Our **facts** table would be the ORDERS table. It will contain all information related to orders (quantity, price, dates).
We then have our **dimension** tables. CUSTOMERS is one, containing all customer details and contact information. PRODUCTS contains the product line, product code, recommended price.

### Customers dimension table

#### Creating a primary key and verifying unicity

In [6]:
sales_df['CUSTOMERNAME'].drop_duplicates().count()

np.int64(92)

In [7]:
sales_df['CUSTOMERNAME'] = sales_df['CUSTOMERNAME'].apply(lambda s: s.strip())

In [8]:
sales_df[['CUSTOMERNAME',
           'PHONE',
       'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE',
       'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'COUNTRY', 'TERRITORY',
       ]].drop_duplicates().count()

CUSTOMERNAME        92
PHONE               92
ADDRESSLINE1        92
ADDRESSLINE2         9
CITY                92
STATE               46
POSTALCODE          89
CONTACTLASTNAME     92
CONTACTFIRSTNAME    92
COUNTRY             92
TERRITORY           54
dtype: int64

In [9]:
customers_df = sales_df[['CUSTOMERNAME', 'PHONE',
       'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE',
       'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'COUNTRY', 'TERRITORY',
       ]].sort_values("CUSTOMERNAME").drop_duplicates()

In [10]:
# This action allows us to create an index for customers
customers_df = customers_df.reset_index(drop=True).reset_index(names="CUSTOMER_ID")

# We create a {customer_name: customer_id} dictionary to create an id column in sales
customer_id_dict = dict()
for x in customers_df[["CUSTOMERNAME", "CUSTOMER_ID"]].values:
    customer_id_dict[x[0]] = x[1]

sales_df["CUSTOMER_ID"] = sales_df["CUSTOMERNAME"].apply(lambda name: customer_id_dict[name])

For each customer name (which seems to be a business entity), there is a contact name. There does not seem to be multiple contacts per customer names. We can link a unique key to the contact name – we don't have to cross multiple informations for it.

Before that, we notice missing values in the Territory field which is useful to our analysis. There are other missing values in State, or Postal Codes, but as of now they are not very relevant.

#### Territory – infering missing values and cleaning

In [11]:
customers_df[["COUNTRY", "TERRITORY"]].groupby(["COUNTRY", "TERRITORY"], dropna=False).count()

,
COUNTRY,TERRITORY
Australia,APAC
Austria,EMEA
Belgium,EMEA
Canada,NaN
Denmark,EMEA
Finland,EMEA
France,EMEA
Germany,EMEA
Ireland,EMEA


In [12]:
# "Japan" has been used as a territory when it should be a region, APAC.
# "NA" for North America should be used as a territory for USA and Canada. 
# We replace remaining nans with "NA". We can even wonder if the value "NA" hasn't been misinterpreted as nan!
# There seems to be no "LATAM" clients.

customers_df["TERRITORY"] = customers_df["TERRITORY"].apply(lambda x: "APAC" if x == "Japan" else x).fillna("NA")

### Products dimension table

In [13]:
products_df = sales_df[['PRODUCTLINE', 'MSRP', 'PRODUCTCODE',]] \
    .drop_duplicates() \
    .sort_values("PRODUCTCODE") \
    .reset_index(drop=True) \
    .reset_index(names="PRODUCT_ID")

In [14]:
# We create a {product_code: product_id} dictionary to create an id column in sales
product_id_dict = dict()
for x in products_df[["PRODUCTCODE", "PRODUCT_ID"]].values:
    product_id_dict[x[0]] = x[1]

sales_df["PRODUCT_ID"] = sales_df["PRODUCTCODE"].apply(lambda code: product_id_dict[code])

### Orders fact table

In [24]:
orders_df = sales_df.drop(list(products_df.columns.values[1:]) + list(customers_df.columns.values[1:]), axis=1)

In [27]:
orders_df = orders_df.reset_index(names="ORDER_ID")

In [29]:
orders_df['ORDERDATE'] = pd.to_datetime(orders_df['ORDERDATE'])

In [33]:
# This allows us to order months more easily in dashboards. Instead of having 1, 11, 12, 2, it orders it automatically
# as 01, 02..., 09, 10, 11, 12
orders_df["MONTH_ID"] = orders_df["MONTH_ID"].apply(lambda x: str(x).zfill(2))

## Upload data to duckdb and define schemas

In [40]:
orders_df.columns

Index(['ORDER_ID', 'ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH',
       'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID',
       'YEAR_ID', 'DEALSIZE', 'CUSTOMER_ID', 'PRODUCT_ID'],
      dtype='str')

In [ ]:
con = duckdb.connect()

con.execute("""
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    customer_name VARCHAR,
    phone VARCHAR,
    address_line_1 VARCHAR,
    address_line_2 VARCHAR,
    city VARCHAR,
    state VARCHAR,
    postal_code VARCHAR,
    contact_last_name VARCHAR,
    contact_first_name VARCHAR,
    country VARCHAR,
    territory VARCHAR
);
            
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_line VARCHAR,
    recommended_price INT,
    product_code VARCHAR
);
            
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    order_number INT,
    quantity_ordered INT,
    price_each FLOAT,
    order_line_number INT,
    sales FLOAT,
    order_date DATE,
    status VARCHAR,
    quarter INT,
    month VARCHAR,
    year INT,
    dealsize VARCHAR,
    customer_id INT REFERENCES customers (customer_id),
    product_id INT REFERENCES products (product_id)
);
""")

In [41]:
con.register("customers_temp", customers_df)
con.register("products_temp", products_df)
con.register("orders_temp", orders_df)

# insert avec cast explicite (important)
con.execute("""
INSERT INTO customers
SELECT 
    CAST(CUSTOMER_ID AS INTEGER) AS customer_id,
    CAST(CUSTOMERNAME AS VARCHAR) AS customer_name,
    CAST(PHONE AS VARCHAR) AS phone,
    CAST(ADDRESSLINE1 AS VARCHAR) AS address_line_1,
    CAST(ADDRESSLINE2 AS VARCHAR) AS address_line_2,
    CAST(CITY AS VARCHAR) AS city,
    CAST(STATE AS VARCHAR) AS state,
    CAST(POSTALCODE AS VARCHAR) AS postal_code,
    CAST(CONTACTLASTNAME AS VARCHAR) AS contact_last_name,
    CAST(CONTACTFIRSTNAME AS VARCHAR) AS contact_first_name,
    CAST(COUNTRY AS VARCHAR) AS country,
    CAST(TERRITORY AS VARCHAR) AS territory
FROM customers_temp;
            
INSERT INTO products
SELECT
    CAST(PRODUCT_ID AS INTEGER) AS product_id,
    CAST(PRODUCTLINE AS VARCHAR) AS product_line,
    CAST(MSRP AS INTEGER) AS recommended_price,
    CAST(PRODUCTCODE AS VARCHAR) AS product_code
FROM products_temp;
            
INSERT INTO orders
SELECT
    CAST(ORDER_ID AS INTEGER) AS order_id,
    CAST(ORDERNUMBER AS INTEGER) AS order_number,
    CAST(QUANTITYORDERED AS INTEGER) AS quantity_ordered,
    CAST(PRICEEACH AS FLOAT) AS price_each,
    CAST(ORDERLINENUMBER AS INTEGER) AS order_line_number,
    CAST(SALES AS FLOAT) AS sales,
    CAST(ORDERDATE AS DATE) AS order_date,
    CAST(STATUS AS VARCHAR) AS status,
    CAST(QTR_ID AS INTEGER) AS quarter,
    CAST(MONTH_ID AS VARCHAR) AS month,
    CAST(YEAR_ID AS INTEGER) AS year,
    CAST(DEALSIZE AS VARCHAR) AS dealsize,
    CAST(CUSTOMER_ID AS INTEGER) AS customer_id,
    CAST(PRODUCT_ID AS INTEGER) AS product_id
FROM orders_temp;
""")

# Exploratory Data Analysis

Now that we have a structured database on SQL, we can start to explore the contents of this database.
I choose to use SQL here, but we could have used pandas.

## Exploratory queries

In [ ]:
# Top countries for sales ?
con.execute("""
SELECT 
    c.country,
    SUM(o.sales) AS total_revenue_dollars
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.country
ORDER BY total_revenue_dollars DESC;
""").df()

,country,total_revenue_dollars
0,USA,3.627983e+06
1,Spain,1.215687e+06
2,France,1.110917e+06
3,Australia,6.306231e+05
4,UK,4.788805e+05
5,Italy,3.746743e+05
6,Finland,3.295819e+05
7,Norway,3.074637e+05
8,Singapore,2.884884e+05
9,Denmark,2.456372e+05


In [55]:
# Top 5 clients ?
con.execute("""
WITH top_5_clients as (SELECT 
    c.customer_name,
    SUM(o.sales) AS total_spent_dollars,
    COUNT(DISTINCT o.order_number) as order_amount
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_name
ORDER BY total_spent_dollars DESC
LIMIT 5)
            
SELECT *,
    total_spent_dollars / order_amount as average_order_spend
FROM top_5_clients;
""").df()

,customer_name,total_spent_dollars,order_amount,average_order_spend
0,Euro Shopping Channel,912294.110474,26,35088.235018
1,Mini Gifts Distributors Ltd.,654858.058105,17,38521.062241
2,"Australian Collectors, Co.",200995.410156,5,40199.082031
3,Muscle Machine Inc,197736.940186,4,49434.235046
4,La Rochelle Gifts,180124.899719,4,45031.224930


In [58]:
# Top produit par catégorie
con.execute("""
SELECT product_line,
        product_code as top_product_code,
        revenue as product_revenue
FROM (
    SELECT 
        p.product_line,
        p.product_code,
        SUM(o.sales) AS revenue,
        RANK() OVER (
            PARTITION BY p.product_line 
            ORDER BY SUM(o.sales) DESC
        ) AS rank_in_category
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.product_line, p.product_code
) t
WHERE rank_in_category = 1;
""").df()

,product_line,top_product_code,product_revenue
0,Ships,S24_2011,114032.128906
1,Motorcycles,S10_4698,170401.070923
2,Planes,S18_1662,139421.969849
3,Trains,S18_3259,90457.039673
4,Vintage Cars,S18_1749,127310.419678
5,Trucks and Buses,S12_1666,136692.719238
6,Classic Cars,S18_3232,288245.420898
